# 04_OPTIMIZACIÓN

In [ ]:
#Volvemos a realizar la lectura de los datos
#Lectura de los dataset y volvemos a separar
import pandas as pd
train=pd.read_csv(r"HEART\data\processed\train.csv")#cambiar ruta.
test=pd.read_csv(r"\HEART\data\processed\test.csv")
X_train=train.drop(columns='target')
y_train=train['target']
X_test=test.drop(columns='target')
y_test=test['target']

In [15]:
#Definimos de nuevo el preprocesador:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,cross_validate,GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [16]:
numericas=X_train.drop(columns=('Gender')).columns.tolist()
preprocessor=ColumnTransformer([("num",StandardScaler(),numericas)],remainder='passthrough')

In [17]:
#Ahora vayamos a definir la pipeline pero utilizando GridSearch:
pipeline_xgb=Pipeline([("proc",preprocessor),("model",XGBClassifier(random_state=42,eval_metric='logloss'))])

param_grid={"model__n_estimators":[100,200,300],
            "model__max_depth":[3,5,7,9],
            "model__learning_rate":[0.01,0.05,0.1]}

#N_estimators: numero de arboles en cadena que se introducen.
#Max_depth: numero de divisiones maximas que permiten a cada árbol. Mas divisiones
#permiten división mas compleja pero mas sobreajuste
#Learning_rate: tasa de aprendizaje entre distintos árboles.


In [18]:
cv=StratifiedKFold(n_splits=5,random_state=42,shuffle=True)
grid=GridSearchCV(pipeline_xgb,param_grid=param_grid,cv=cv,scoring='roc_auc',n_jobs=-1)
grid.fit(X_train,y_train)
#n_jobs=-1 utiliza todos los procesadores disponibles

print(f"Mejores parametros:{grid.best_params_}")
print(f"ROC_AUC:{round(grid.best_score_,3)}")

Mejores parametros:{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}
ROC_AUC:0.997


In [21]:
#Mejor modelo:
from sklearn.metrics import classification_report
mejor_modelo=grid.best_estimator_
y_pred=mejor_modelo.predict(X_test)
y_proba=mejor_modelo.predict_proba(X_test)[:,1]
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.97      0.98       101
           1       0.98      1.00      0.99       162

    accuracy                           0.99       263
   macro avg       0.99      0.99      0.99       263
weighted avg       0.99      0.99      0.99       263



In [22]:
#guardar el modelo en un joblib.
import joblib
joblib.dump(mejor_modelo, "../models/xgboost_heart.joblib")

['../models/xgboost_heart.joblib']